# 🤖 Module 5: Build a Real Agent (Tool Calling + RAG + Memory)

This notebook combines everything:
- **RAG** (Module 4) — ChromaDB knowledge base
- **Tool Calling** (Module 3) — weather, calculator, web search
- **Conversation Memory** — multi-turn chat
- **Real LLM** — Groq (free API)

> Get your free Groq key at: https://console.groq.com

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q sentence-transformers chromadb groq
print('✅ Done')

---
## Step 2: Imports & API Key

In [ ]:
import json
from groq import Groq
from sentence_transformers import SentenceTransformer
import chromadb

GROQ_API_KEY = 'your_key_here'  # paste from console.groq.com
MODEL = 'llama-3.1-8b-instant'

client = Groq(api_key=GROQ_API_KEY)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Ready')

---
## Step 3: Build the Knowledge Base (RAG Component)

In [ ]:
# Company documents
DOCS = [
    {'id':'d1','source':'refund_policy.txt','text':'Customers can return products within 30 days. Items must be in original condition. Digital products cannot be returned.'},
    {'id':'d2','source':'refund_policy.txt','text':'Refunds are processed in 5 to 7 business days. Contact support@company.com to initiate a return. Refunds go to the original payment method.'},
    {'id':'d3','source':'shipping.txt','text':'Standard shipping takes 3 to 5 business days. Express shipping delivers in 1 to 2 days for an extra fee. Free shipping on orders above $50.'},
    {'id':'d4','source':'shipping.txt','text':'International shipping takes 7 to 14 business days. Tracking info is emailed once the order ships. Orders are dispatched within 24 hours of payment.'},
    {'id':'d5','source':'pricing.txt','text':'All prices include taxes. We offer a price match guarantee. Contact sales@company.com with proof of a lower price elsewhere.'},
]

# Store in ChromaDB
chroma = chromadb.Client()
col = chroma.create_collection('docs', metadata={'hnsw:space':'cosine'})
col.add(
    ids=[d['id'] for d in DOCS],
    embeddings=embedder.encode([d['text'] for d in DOCS]).tolist(),
    documents=[d['text'] for d in DOCS],
    metadatas=[{'source':d['source']} for d in DOCS]
)
print(f'✅ Stored {col.count()} chunks in ChromaDB')

---
## Step 4: Define All Tools

In [ ]:
def search_knowledge_base(query: str) -> str:
    """Search company documents (RAG)."""
    qvec = embedder.encode(query).tolist()
    res = col.query(query_embeddings=[qvec], n_results=2, include=['documents','metadatas'])
    chunks = res['documents'][0]
    sources = [m['source'] for m in res['metadatas'][0]]
    return '\n'.join([f'[{s}] {c}' for s, c in zip(sources, chunks)])

def get_weather(city: str) -> str:
    """Simulated weather (replace with real API)."""
    db = {'dhaka':'32C Humid','london':'15C Rainy','tokyo':'22C Clear','paris':'18C Cloudy'}
    return db.get(city.lower(), f'No data for {city}')

def calculate(a: float, b: float, operation: str) -> str:
    """Arithmetic calculator."""
    ops = {'add':a+b,'subtract':a-b,'multiply':a*b,'divide':a/b if b else 'Error: div by zero'}
    return str(ops.get(operation, 'Unknown operation'))

def search_web(query: str) -> str:
    """Simulated web search."""
    return f'Web search result for "{query}": This is a simulated result. In production, connect to a real search API.'

TOOL_REGISTRY = {
    'search_knowledge_base': search_knowledge_base,
    'get_weather': get_weather,
    'calculate': calculate,
    'search_web': search_web,
}

def execute_tool(name, args):
    if name not in TOOL_REGISTRY:
        return f'Error: tool {name} not found'
    try:
        return str(TOOL_REGISTRY[name](**args))
    except Exception as e:
        return f'Error: {e}'

# Quick test
print('RAG test:', search_knowledge_base('refund time')[:80])
print('Calc test:', calculate(150, 3.5, 'multiply'))

---
## Step 5: Tool Schemas (What the LLM Reads)

In [ ]:
TOOL_SCHEMAS = [
  {'name':'search_knowledge_base',
   'description':'Search company documents and policies. Use this for ANY company-specific question.',
   'parameters':{'type':'object','properties':{'query':{'type':'string','description':'search query'}},'required':['query']}},
  {'name':'get_weather',
   'description':'Get current weather for a city.',
   'parameters':{'type':'object','properties':{'city':{'type':'string','description':'city name'}},'required':['city']}},
  {'name':'calculate',
   'description':'Perform math: add, subtract, multiply, divide.',
   'parameters':{'type':'object','properties':{
     'a':{'type':'number'},'b':{'type':'number'},
     'operation':{'type':'string','description':'add|subtract|multiply|divide'}
   },'required':['a','b','operation']}},
  {'name':'search_web',
   'description':'Search the web for general information not in company documents.',
   'parameters':{'type':'object','properties':{'query':{'type':'string'}},'required':['query']}},
]

GROQ_TOOLS = [{'type':'function','function':s} for s in TOOL_SCHEMAS]
print(f'✅ {len(TOOL_SCHEMAS)} tools registered')

---
## Step 6: The Complete Agent Loop

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant for TechCorp customer service.
You have access to: search_knowledge_base, get_weather, calculate, search_web.
Rules:
1. ALWAYS use search_knowledge_base for company policy questions.
2. Use calculate for any math.
3. Never invent company information - retrieve it.
4. Be concise and clear."""

class Agent:
    def __init__(self):
        self.history = [{'role':'system','content':SYSTEM_PROMPT}]

    def chat(self, user_message: str, max_steps: int = 10) -> str:
        """Process one user message with full tool-calling loop."""
        print(f'\n{"="*60}')
        print(f'USER: {user_message}')
        self.history.append({'role':'user','content':user_message})

        for step in range(max_steps):
            response = client.chat.completions.create(
                model=MODEL,
                messages=self.history,
                tools=GROQ_TOOLS,
                tool_choice='auto',
                temperature=0
            )
            msg = response.choices[0].message

            # Final answer
            if not msg.tool_calls:
                self.history.append({'role':'assistant','content':msg.content})
                print(f'AGENT: {msg.content}')
                return msg.content

            # Tool calls
            self.history.append(msg)
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                print(f'  → calls {name}({args})')
                result = execute_tool(name, args)
                print(f'    result: {result[:100]}')
                self.history.append({
                    'role':'tool',
                    'tool_call_id':tc.id,
                    'content':result
                })

        return 'Error: max steps reached'

    def reset(self):
        """Clear conversation history."""
        self.history = [{'role':'system','content':SYSTEM_PROMPT}]
        print('✅ Memory cleared')

print('✅ Agent class ready')

---
## Step 7: Test Single-Tool Queries

In [ ]:
if GROQ_API_KEY == 'your_key_here':
    print('⚠️  Paste your Groq API key in Step 2 to run this cell.')
else:
    agent = Agent()

    # Test RAG
    agent.chat('What is the refund policy?')

    # Test calculator
    agent.chat('What is 150 multiplied by 3.5?')

    # Test weather
    agent.chat('What is the weather in Tokyo?')

---
## Step 8: Test Multi-Tool Query (The Real Power)

In [ ]:
if GROQ_API_KEY == 'your_key_here':
    print('⚠️  Paste your Groq API key in Step 2 to run this cell.')
else:
    agent.reset()

    # This requires TWO tools: RAG + calculator
    agent.chat('What is our refund policy and what is 89 times 12?')

    # This requires TWO tools: RAG + weather
    agent.chat('What is the shipping time to London and what is the weather there?')

---
## Step 9: Test Conversation Memory (Multi-Turn)

In [ ]:
if GROQ_API_KEY == 'your_key_here':
    print('⚠️  Paste your Groq API key in Step 2 to run this cell.')
else:
    agent.reset()

    agent.chat('What is the refund policy?')
    agent.chat('How long does that take?')          # agent knows 'that' = refund
    agent.chat('And what about shipping time?')      # agent knows context = our company

---
## ✅ What You Built

| Component | Role |
|---|---|
| ChromaDB + SentenceTransformer | Knowledge base (RAG) |
| `get_weather`, `calculate`, `search_web` | External tool functions |
| Tool schemas (JSON) | Tell the LLM what tools exist |
| `execute_tool()` | Runs the right function safely |
| `Agent` class | Holds memory + runs the loop |
| Groq Llama-3 | The brain making all decisions |

### This is a production-ready agent pattern.
To make it truly production-ready, add:
- Real weather API (OpenWeatherMap)
- Real web search (SerpAPI, Tavily)
- Persistent vector DB (not in-memory)
- Error logging and observability

**Next → Module 6: Multi-Agent Systems** (orchestrator + specialist workers)